# Physics

**What this shows:** the physics options of SWAN available in the `PHYSICS` component:
source-term packages, whitecapping and swell dissipation, breaking, bottom friction,
triads and the processes for special cases.

**Prerequisites:** [Tutorial 5: Choosing model settings](../tutorial/05_model_settings.ipynb).

**You will learn:**

- which source-term packages `GEN3` offers, and what they mean
- the options for depth-induced breaking, bottom friction and triads
- how to add set-up, diffraction, obstacles and vegetation, and how to switch
  processes off

**Data used:** none.

Each option below links to its section of the
[SWAN user manual](https://swanmodel.sourceforge.io/online_doc/swanuse/node28.html).
[Physics sensitivity](physics_sensitivity.ipynb) runs SWAN with several packages and
compares the results.

## Setup

In [1]:
from rompy_swan.components.group import PHYSICS
from rompy_swan.components.physics import (
    BREAKING_BKD,
    BREAKING_CONSTANT,
    DIFFRACTION,
    FRICTION_COLLINS,
    FRICTION_JONSWAP,
    FRICTION_MADSEN,
    GEN3,
    OBSTACLE,
    OBSTACLES,
    OFF,
    OFFS,
    QUADRUPL,
    SETUP,
    SSWELL_ZIEGER,
    TRIAD,
    TRIAD_DCTA,
    VEGETATION,
)
from rompy_swan.subcomponents.physics import (
    JANSSEN,
    KOMEN,
    LINE,
    REFL,
    ST6C1,
    TRANSM,
    WESTHUYSEN,
)

## 1. Wind input and whitecapping: `GEN3`

`GEN3` switches on third-generation physics. Its `source_terms` choose the package of
wind input and whitecapping formulations:

| Package | Formulations | Notes |
|---|---|---|
| `WESTHUYSEN()` | Yan wind input, saturation-based whitecapping | SWAN default since 41.45 |
| `KOMEN()` | WAM cycle 3 | SWAN default before 41.45 |
| `JANSSEN()` | WAM cycle 4 | not recommended by the SWAN team |
| `ST6C1()` … `ST6C5()` | Rogers et al. (2012) with calibrated coefficient sets | pair with `SSWELL_ZIEGER` |

rompy-swan also writes Wu's (1982) drag formulation, `DRAG WU`, SWAN's default.

In [2]:
for terms in [WESTHUYSEN(), KOMEN(), JANSSEN(), ST6C1()]:
    print(GEN3(source_terms=terms).render())

GEN3 WESTHUYSEN DRAG WU
GEN3 KOMEN DRAG WU
GEN3 JANSSEN DRAG WU
GEN3 ST6 a1sds=4.7e-07 a2sds=6.6e-06 p1sds=4.0 p2sds=4.0 UP HWANG VECTAU U10PROXY windscaling=28.0 AGROW


The ST6 presets fill in the coefficients of each calibration. ST6 is meant to be used
with swell dissipation (`SSWELL ZIEGER`), which the presets do not add:

In [3]:
print(PHYSICS(gen=GEN3(source_terms=ST6C1()), sswell=SSWELL_ZIEGER()).render())

GEN3 ST6 a1sds=4.7e-07 a2sds=6.6e-06 p1sds=4.0 p2sds=4.0 UP HWANG VECTAU U10PROXY windscaling=28.0 AGROW

SSWELL ZIEGER


## 2. Quadruplets

Four-wave interactions move energy within the spectrum and are on by default with
`GEN3`. `QUADRUPL` changes the method (`iquad`) and coefficients. Without wind, SWAN
requires them to be switched off with `OFF` (see section 6).

In [4]:
print(QUADRUPL(iquad=2, lambd=0.25).render())

QUADRUPL iquad=2 lambda=0.25


## 3. Depth-induced breaking

Breaking is on by default in SWAN, with a constant breaker index.

- `BREAKING_CONSTANT(alpha, gamma)`: Battjes and Janssen (1978), `gamma=0.73`;
- `BREAKING_BKD`: Salmon et al. (2015), with a breaker index that depends on the
  bottom slope and wave steepness, recommended for varying bathymetry.

In [5]:
print(BREAKING_CONSTANT(alpha=1.0, gamma=0.73).render())
print(BREAKING_BKD(alpha=1.0, gamma0=0.54, a1=7.59, a2=-8.06, a3=8.09).render())

BREAKING CONSTANT alpha=1.0 gamma=0.73
BREAKING BKD alpha=1.0 gamma0=0.54 a1=7.59 a2=-8.06 a3=8.09


## 4. Bottom friction

Bottom friction is **off** unless it is set:

- `FRICTION_JONSWAP(cfjon)`: empirical, `cfjon=0.038` m²/s³ (0.019 for sandy
  beaches with swell);
- `FRICTION_COLLINS(cfw)`: drag law with a friction factor;
- `FRICTION_MADSEN(kn)`: eddy-viscosity model with a roughness length `kn`.

In [6]:
for friction in [FRICTION_JONSWAP(cfjon=0.038), FRICTION_COLLINS(cfw=0.015), FRICTION_MADSEN(kn=0.05)]:
    print(friction.render())

FRICTION JONSWAP CONSTANT cfjon=0.038
FRICTION COLLINS cfw=0.015
FRICTION MADSEN kn=0.05


A spatially varying friction coefficient is read from an input grid (`var="friction"`,
see [Input grids](input_grids.ipynb)).

## 5. Triads

Three-wave interactions transfer energy to higher frequencies in shallow water. They
are **off** unless set. `TRIAD` uses SWAN's default method (DCTA since 41.45);
`TRIAD_DCTA` exposes its coefficients.

In [7]:
print(TRIAD().render())
print(TRIAD_DCTA(trfac=4.4).render())

TRIAD
TRIAD DCTA trfac=4.4 COLL


## 6. Special processes

| Component | Process |
|---|---|
| `SETUP()` | wave-induced set-up (only on open coasts, not in closed basins) |
| `DIFFRACTION()` | phase-decoupled diffraction approximation |
| `OBSTACLES([OBSTACLE(...)])` | transmission and reflection at breakwaters and dams |
| `VEGETATION(...)` | dissipation by vegetation (heights, diameters, densities per layer) |
| `OFFS([OFF(physics=...)])` | switch off a default process: `windgrowth`, `quadrupl`, `wcapping`, `breaking`, `refrac`, `fshift`, `bndchk` |

In [8]:
breakwater = OBSTACLE(
    transmission=TRANSM(trcoef=0.3),
    reflection=REFL(reflc=0.5),
    line=LINE(xp=[115.735, 115.745], yp=[-32.065, -32.055]),
)
physics = PHYSICS(
    gen=GEN3(),
    breaking=BREAKING_BKD(alpha=1.0, gamma0=0.54, a1=7.59, a2=-8.06, a3=8.09),
    friction=FRICTION_JONSWAP(cfjon=0.038),
    triad=TRIAD(),
    setup=SETUP(),
    diffraction=DIFFRACTION(),
    obstacle=OBSTACLES(obstacles=[breakwater]),
    vegetation=VEGETATION(iveg=1, height=[0.5], diamtr=[0.01], drag=[1.0], nstems=[400]),
    deactivate=OFFS(offs=[OFF(physics="refrac")]),
)
print(physics.render())

GEN3 WESTHUYSEN DRAG WU

BREAKING BKD alpha=1.0 gamma0=0.54 a1=7.59 a2=-8.06 a3=8.09

FRICTION JONSWAP CONSTANT cfjon=0.038

TRIAD

VEGETATION iveg=1 height=0.5 diamtr=0.01 nstems=400 drag=1.0

OBSTACLE TRANSM trcoef=0.3 REFL reflc=0.5 LINE 115.735 -32.065 115.745 -32.055

SETUP

DIFFRACTION

OFF REFRAC


This combination is only to show the syntax; switching refraction off, for example,
only makes sense for tests.

## Summary

- `GEN3(source_terms=...)` chooses the wind input and whitecapping package;
  WESTHUYSEN is SWAN's current default.
- Breaking is on by default; friction and triads must be switched on for coastal
  models.
- Set-up, diffraction, obstacles and vegetation are added when needed, and `OFF`
  removes default processes.